#作业2-1多元线性回归

**题目要求手搓的是C++里的回归类**（LinearRegression），不是预处理。

数据预处理我用Python在notebook里一步步写（读csv、划分、标准化），因为：
-ocean_proximity是文字，C++读起来麻烦
-标准化公式简单，用Python先弄好，C++只读数字csv省事

流程：housing.csv -> 预处理 -> train.csv/test.csv -> C++训练


In [ ]:
#第1步：读原始housing.csv
import csv
import random
import math
import os
os.makedirs('data',exist_ok=True)
FEAT=['longitude','latitude','housing_median_age','total_rooms','total_bedrooms','population','households','median_income']
rows=[]
with open('data/housing.csv',newline='',encoding='utf-8') as f:
    reader=csv.DictReader(f)
    for row in reader:
        try:
            x=[float(row[c]) for c in FEAT]
            y=float(row['median_house_value'])/100000.0#房价改成十万美元，数字小
            rows.append(x+[y])
        except:
            pass#坏行跳过
print('读入',len(rows),'条')


In [ ]:
#第2步：total_bedrooms有空值，用该列中位数填（书上常见做法）
for j in range(len(FEAT)):
    vals=[r[j] for r in rows if not math.isnan(r[j])]
    med=sorted(vals)[len(vals)//2]
    for r in rows:
        if math.isnan(r[j]):
            r[j]=med
print('缺失值填完了')


In [ ]:
#第3步：8:2划分训练集和测试集
random.seed(42)
random.shuffle(rows)
n=len(rows)
n_train=int(n*0.8)
train=rows[:n_train]
test=rows[n_train:]
print('训练',len(train),'测试',len(test))


In [ ]:
#第4步：标准化，(x-mean)/std，只用训练集算mean和std
n_feat=len(FEAT)
means=[sum(r[i] for r in train)/len(train) for i in range(n_feat)]
stds=[]
for i in range(n_feat):
    v=sum((r[i]-means[i])**2 for r in train)/len(train)
    stds.append(math.sqrt(v) if v>1e-12 else 1.0)
def norm(data):
    out=[]
    for r in data:
        x=[(r[i]-means[i])/stds[i] for i in range(n_feat)]
        out.append(x+[r[-1]])#最后一列y不标准化
    return out
train=norm(train)
test=norm(test)
print('标准化完成')


In [ ]:
#第5步：写到csv，C++就读这两个文件
def save(path,data):
    with open(path,'w',newline='') as f:
        w=csv.writer(f)
        for r in data:
            w.writerow(r)
save('data/train.csv',train)
save('data/test.csv',test)
print('已保存 data/train.csv 和 data/test.csv')


In [ ]:
#看C++源码
with open('linear_regression.cpp',encoding='utf-8') as f:
    print(f.read())


In [ ]:
!g++ -std=c++11 linear_regression.cpp -o lr
!./lr


In [ ]:
import matplotlib.pyplot as plt
import csv
yt,yp=[],[]
with open('data/pred_linear.csv') as f:
    for row in csv.DictReader(f):
        yt.append(float(row['y_true']))
        yp.append(float(row['y_pred']))
plt.scatter(yt,yp,s=8,alpha=0.5)
plt.plot([min(yt),max(yt)],[min(yt),max(yt)],'r--')
plt.xlabel('true')
plt.ylabel('pred')
plt.title('linear')
plt.show()


##小结
-手搓重点：linear_regression.cpp里的类和梯度下降
-预处理：上面Python单元格，没用sklearn
-MSE看终端输出
